# Learned Index Demo

This project compares two ways to search through a sorted list of numbers:

- **Binary Search** – the classic method. Keeps cutting the list in half until it finds the value.
- **Learned Index** – uses a simple ML model (Linear Regression) to guess where the value probably is, then only searches a small area around that guess.

The idea comes from a 2018 Google research paper by Kraska et al.

| | Binary Search | Learned Index |
|---|---|---|
| How it works | Divide and conquer | Predict then verify |
| Needs training | No | Yes (once) |
| Speed | O(log n) | O(1) predict + tiny search |

## 1. Install libraries

In [ ]:
%pip install numpy scikit-learn matplotlib

## 2. Imports

In [ ]:
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")  # needed for notebooks
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

print("ready")

## 3. Generate the dataset

We create 100,000 unique sorted integers between 0 and 1,000,000.
The fixed seed means you get the same numbers every time you run this.

In [ ]:
def generate_dataset(size=100_000):
    rng  = np.random.default_rng(seed=21)           # fixed seed
    data = rng.integers(0, 1_000_000, size=size * 2) # generate extra in case of duplicates
    data = np.unique(data)                           # drop duplicates
    data = np.sort(data[:size])                      # keep exactly size values
    return data

data = generate_dataset()

print(f"size  : {len(data):,}")
print(f"range : {data[0]} to {data[-1]:,}")
print(f"sample: {data[:8]}")

## 4. Binary search

Standard binary search. Looks at the middle element, then narrows down left or right.
Takes at most ~17 steps for 100,000 elements.

In [ ]:
def binary_search(arr, target):
    low, high = 0, len(arr) - 1

    while low <= high:
        mid = (low + high) // 2

        if arr[mid] == target:
            return mid          # found
        elif arr[mid] < target:
            low = mid + 1       # go right
        else:
            high = mid - 1      # go left

    return -1  # not found

# quick test
key = data[42_000]
print(f"searching {key} -> index {binary_search(data, key)}  (expected 42000)")

## 5. Learned index

Since the data is sorted, the key value and its position are almost perfectly linear:

    index ≈ slope × key + intercept

We fit a Linear Regression model on this. At search time:
1. The model predicts roughly where the key should be.
2. We search only a small window around that spot (±max_error).

The window is usually just a few hundred elements instead of 100,000.

In [ ]:
class LearnedIndex:

    def __init__(self):
        self.model     = LinearRegression()
        self.max_error = 0
        self.data      = None

    def train(self, data):
        self.data = data

        X = data.reshape(-1, 1)       # keys as input
        Y = np.arange(len(data))      # positions as output

        self.model.fit(X, Y)

        # find the worst prediction error on training data
        predicted      = self.model.predict(X).astype(int)
        errors         = np.abs(predicted - Y)
        self.max_error = int(np.max(errors))

        slope     = self.model.coef_[0]
        intercept = self.model.intercept_

        print(f"equation  : index = {slope:.4f} * key + {intercept:.1f}")
        print(f"max error : {self.max_error} positions")
        print(f"window    : {2 * self.max_error + 1} elements (out of 100,000)")

    def search(self, target):
        # predict position using the model
        predicted_pos = int(round(self.model.predict([[target]])[0]))

        # define a small search window
        lo = max(0, predicted_pos - self.max_error)
        hi = min(len(self.data) - 1, predicted_pos + self.max_error)

        # binary search inside that window only
        result = binary_search(self.data[lo : hi + 1], target)

        if result == -1:
            return -1, predicted_pos, lo, hi

        return lo + result, predicted_pos, lo, hi


print("training...")
t0 = time.perf_counter()
li = LearnedIndex()
li.train(data)
print(f"done in {time.perf_counter() - t0:.3f}s")

## 6. Search a single value

Change `TARGET` to any number from the dataset and see both algorithms find it.

In [ ]:
TARGET = int(data[55_000])  # change this if you want

bs_idx               = binary_search(data, TARGET)
li_idx, pred, lo, hi = li.search(TARGET)

print(f"target         : {TARGET}")
print(f"binary search  : index {bs_idx}")
print(f"learned index  : index {li_idx}")
print(f"  model guess  : {pred}")
print(f"  search window: [{lo}, {hi}]  ({hi - lo + 1} elements)")
print()
if bs_idx == li_idx:
    print("both agree")
else:
    print("mismatch - check the code")

## 7. Benchmark — 10,000 lookups

Runs both algorithms on 10,000 random keys and measures total time.

In [ ]:
def run_benchmark(data, learned_index, n=10_000):
    rng     = np.random.default_rng(seed=99)
    queries = rng.choice(data, size=n, replace=False)  # pick keys that exist

    # time binary search
    t0         = time.perf_counter()
    bs_results = [binary_search(data, q) for q in queries]
    bs_time    = time.perf_counter() - t0

    # time learned index
    t0         = time.perf_counter()
    li_results = [learned_index.search(q)[0] for q in queries]
    li_time    = time.perf_counter() - t0

    mismatches = sum(b != l for b, l in zip(bs_results, li_results))
    speedup    = bs_time / li_time if li_time > 0 else float("inf")

    print(f"queries    : {n:,}")
    print(f"mismatches : {mismatches}")
    print()
    print(f"binary search  : {bs_time:.3f}s total  |  {bs_time/n*1e6:.1f} us per lookup")
    print(f"learned index  : {li_time:.3f}s total  |  {li_time/n*1e6:.1f} us per lookup")
    print(f"speedup        : {speedup:.2f}x")

    return {"bs_avg": bs_time/n, "li_avg": li_time/n,
            "bs_total": bs_time, "li_total": li_time,
            "speedup": speedup, "mismatches": mismatches}


stats = run_benchmark(data, li)

## 8. Bar chart — average lookup time

Simple bar chart comparing how long each method takes per lookup.

In [ ]:
labels = ["Binary Search", "Learned Index"]
times  = [stats["bs_avg"] * 1e6, stats["li_avg"] * 1e6]  # convert to microseconds
colors = ["steelblue", "tomato"]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, times, color=colors, width=0.4)

# label each bar
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f"{t:.1f} us",
            ha="center", va="bottom", fontsize=10)

ax.set_title("Average lookup time (microseconds)")
ax.set_ylabel("us per lookup")
ax.set_ylim(0, max(times) * 1.3)
plt.tight_layout()
plt.savefig("lookup_time_comparison.png", dpi=120)
plt.show()
print("saved lookup_time_comparison.png")

## 9. Regression plot

Shows how well the model fits the data.
Blue dots are real positions. The red line is what the model predicts.
If they match closely, the model is accurate.

In [ ]:
# sample 2000 points so the chart renders fast
sample_idx    = np.linspace(0, len(data) - 1, 2000, dtype=int)
sample_keys   = data[sample_idx]
predicted_pos = li.model.predict(sample_keys.reshape(-1, 1))

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(sample_keys, sample_idx, s=2, color="steelblue", alpha=0.5, label="actual position")
ax.plot(sample_keys, predicted_pos, color="tomato", linewidth=2, label="model prediction")

ax.set_title("Key value vs array index")
ax.set_xlabel("key value")
ax.set_ylabel("array index")
ax.legend()
plt.tight_layout()
plt.savefig("regression_plot.png", dpi=120)
plt.show()
print("saved regression_plot.png")

## 10. Model scatter plots — 4 views

Four plots that look at the model from different angles:

- **Top left** — actual vs predicted positions, with the error band shaded
- **Top right** — residuals (how far off each prediction was)
- **Bottom left** — histogram of residuals (should be centred near zero)
- **Bottom right** — predicted vs actual index (should be a straight diagonal line)

In [ ]:
# sample 3000 points
idx       = np.linspace(0, len(data) - 1, 3000, dtype=int)
keys      = data[idx]
actual    = idx
predicted = li.model.predict(keys.reshape(-1, 1))
residuals = actual - predicted
mean_res  = residuals.mean()
std_res   = residuals.std()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Learned Index — model analysis", fontsize=13)

# top left: key vs position
ax1 = axes[0, 0]
ax1.scatter(keys, actual, s=4, color="steelblue", alpha=0.4, label="actual")
ax1.plot(keys, predicted, color="tomato", linewidth=2, label="predicted")
ax1.fill_between(keys,
                 predicted - li.max_error,
                 predicted + li.max_error,
                 color="tomato", alpha=0.1,
                 label=f"error band (±{li.max_error})")
ax1.set_title("Key value vs array index")
ax1.set_xlabel("key")
ax1.set_ylabel("index")
ax1.legend(fontsize=8)

# top right: residuals
ax2 = axes[0, 1]
sc = ax2.scatter(keys, residuals, s=4, c=np.abs(residuals), cmap="RdYlGn_r", alpha=0.6)
ax2.axhline(0,              color="black", linewidth=1, linestyle="--", label="zero")
ax2.axhline( li.max_error,  color="tomato", linewidth=1, linestyle=":", label=f"±{li.max_error}")
ax2.axhline(-li.max_error,  color="tomato", linewidth=1, linestyle=":")
fig.colorbar(sc, ax=ax2, label="|error|")
ax2.set_title("Prediction error per key")
ax2.set_xlabel("key")
ax2.set_ylabel("residual")
ax2.legend(fontsize=8)

# bottom left: residual histogram
ax3 = axes[1, 0]
ax3.hist(residuals, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
ax3.axvline(0,             color="black", linewidth=1.2, linestyle="--", label="zero")
ax3.axvline( li.max_error, color="tomato", linewidth=1, linestyle=":")
ax3.axvline(-li.max_error, color="tomato", linewidth=1, linestyle=":")
ax3.set_title("Distribution of residuals")
ax3.set_xlabel("residual (positions)")
ax3.set_ylabel("count")
# show mean and std in a small box
ax3.text(0.97, 0.95, f"mean={mean_res:.1f}\nstd={std_res:.1f}",
         transform=ax3.transAxes, ha="right", va="top", fontsize=9,
         bbox=dict(fc="white", alpha=0.7))
ax3.legend(fontsize=8)

# bottom right: predicted vs actual
ax4 = axes[1, 1]
ax4.scatter(actual, predicted, s=4, color="seagreen", alpha=0.4, label="data")
line = np.array([actual.min(), actual.max()])
ax4.plot(line, line, color="black", linewidth=1.5, linestyle="--", label="y = x")
r2 = li.model.score(data[idx].reshape(-1, 1), actual)
ax4.set_title("Predicted vs actual index")
ax4.set_xlabel("actual index")
ax4.set_ylabel("predicted index")
# show R2 score
ax4.text(0.05, 0.95, f"R2 = {r2:.5f}",
         transform=ax4.transAxes, ha="left", va="top", fontsize=10,
         bbox=dict(fc="lightyellow", alpha=0.8))
ax4.legend(fontsize=8)

plt.tight_layout()
plt.savefig("learned_model_scatter.png", dpi=120, bbox_inches="tight")
plt.show()
print("saved learned_model_scatter.png")
print(f"R2 score  : {r2:.6f}")
print(f"mean error: {mean_res:.2f}")
print(f"std error : {std_res:.2f}")

## 11. Summary

The key idea: for sorted data, the ML model can predict where a value is in O(1) instead of searching the whole list. It then only checks a small window, making it faster than binary search once trained.

In [ ]:
print("Results")
print(f"  binary search : {stats['bs_avg']*1e6:.1f} us/lookup")
print(f"  learned index : {stats['li_avg']*1e6:.1f} us/lookup")
print(f"  speedup       : {stats['speedup']:.2f}x")
print(f"  mismatches    : {stats['mismatches']}")